# Step 3 · SFT

Three parts, each saved to Google Drive as it goes. They can run in separate sessions: if Colab disconnects, run the setup and settings cells, then the part you were on. Finished work is skipped.

1. **Build the training data:** for each training question, sample a few constrained answers and keep the most fluent one that spells the secret, is clean prose and doesn't name the secret. Saved with only "Secret: X" in the prompt.
2. **Train** a LoRA adapter on that data.
3. **Evaluate:** the trained model answers `val` questions with only "Secret: X", scored like E0.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. Needs the dataset from notebook 02.

Rough timing on a free T4 with the defaults: data about 2 hours, training under an hour, evaluation a few minutes. These are estimates.

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
N_PROMPTS = 2000      # training questions to build data for
SAMPLES = 4           # constrained answers sampled per question
EPOCHS = 2
N_EVAL = 100          # val questions to evaluate on (same as E0)
BATCH_SIZE = 16       # generation batch; lower to 8 if you run out of GPU memory

In [ ]:
import gc, torch
from stego.colab import run_dir
from stego.experiments import baselines as e0, sft
from stego.io import read_jsonl

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
name = MODEL.split("/")[-1]
DATA_DIR = run_dir("data")
E0_DIR = run_dir(f"e0-{CHANNEL}-{name}")
RUN_DIR = run_dir(f"sft-{CHANNEL}-{name}")

examples = read_jsonl(DATA_DIR / f"examples_{CHANNEL}.jsonl")
train_rows = sft.pick_training_examples(examples, N_PROMPTS)
val_rows = [r for r in e0.eval_rows(examples, N_EVAL) if r["split"] == "val"]
print(len(train_rows), "training questions;", len(val_rows), "val questions; run folder:", RUN_DIR)

def free_gpu():
    for v in ("model", "tokenizer"):
        globals().pop(v, None)
    gc.collect(); torch.cuda.empty_cache()

## 1. Build the training data

In [ ]:
from stego.generation.runner import load_model
from stego.generation.constrained import AcrosticVocab

model, tokenizer = load_model(MODEL)
vocab = AcrosticVocab(tokenizer, model.config.vocab_size)
SFT_PATH = sft.build_sft_data(RUN_DIR, train_rows, model, tokenizer, vocab, samples=SAMPLES, batch_size=BATCH_SIZE)

# Fluency references for the val questions, from the original model (needed in part 3).
e0.generate_references(RUN_DIR, val_rows, model, tokenizer, sampled=True, batch_size=BATCH_SIZE)

In [ ]:
records = read_jsonl(SFT_PATH)
print(len(records), "training records")
for r in records[:3]:
    print(r["messages"][0]["content"].splitlines()[-1], "|", r["messages"][1]["content"])
    print(r["messages"][-1]["content"]); print()
free_gpu()

## 2. Train

LoRA on all linear layers, loss on the answer only. Checkpoints go to Drive every 50 steps; rerunning resumes from the latest one.

In [ ]:
model, tokenizer = load_model(MODEL)
ADAPTER = sft.train_sft(RUN_DIR, RUN_DIR / "sft_train.jsonl", model, tokenizer, epochs=EPOCHS)
free_gpu()

## 3. Evaluate

The trained model sees only "Secret: X" and the question. Fluency is scored by the original model (adapter switched off), against the same sampled references as before.

In [ ]:
from IPython.display import Markdown, display

model, tokenizer = sft.load_sft_model(MODEL, RUN_DIR / "adapter")
fluency_refs = {r["example_id"]: r["answer"] for r in read_jsonl(RUN_DIR / "references_sampled.jsonl")}
sft.evaluate(RUN_DIR, val_rows, model, tokenizer, fluency_refs, batch_size=BATCH_SIZE)

display(Markdown(sft.results_table(
    E0_DIR / "scored_no_secret.jsonl", E0_DIR / "scored_prompted.jsonl", E0_DIR / "scored_constrained.jsonl",
    RUN_DIR / "scored_sft.jsonl",
)))

In [ ]:
print(e0.show_examples(RUN_DIR, "sft", n=3))

## Reading the results

- **exact / letter_acc** for `sft` should sit well above `prompted` (which is at the chance floor). The model now spells secrets with no instructions in the prompt.
- **fluency** shows what the hidden channel costs in naturalness.
- E0 rows were scored against E0's own sampled references, so their fluency is comparable but not identical.

Next: step 4, DPO.